# LLM Trading Strategy vs. Momentum vs. Buy-and-Hold

End-to-end example: does a local LLM's judgment add anything over a simple rule or passive holding, on the 8-company universe from the identification benchmark?

**Universe (real tickers):** AMD, APP (AppLovin), GM, KR (Kroger), MCK (McKesson), NFLX (Netflix), PLTR (Palantir), SNDK (SanDisk) — prices from yfinance.

**Three arms, monthly rebalance, long-only top-3:**
1. **LLM** — a served local model (`qwen35-35b-a3b`, the best speed×capability from the ID benchmark) scores the stocks each month; we hold the top 3.
2. **Momentum** — the *same* framework with a deterministic momentum rule instead of the LLM (isolates the LLM's contribution).
3. **Buy & Hold** — equal-weight the eligible names, adding new listings as they IPO.

**No hindsight:** the LLM sees the stocks **anonymized** (`S1..Sn`, feature-only, no company names) — otherwise it would use training-time knowledge of who won (AMD/Netflix/Palantir), i.e. look-ahead. Features at date *t* use only prices ≤ *t*; positions earn from *t+1*.

> ⚠️ Illustrative only: 8 concentrated names, one 7-year window (universe grows as names list), single run, flat 5bps costs, no significance testing. **Prereq:** native llama.cpp built + the model downloaded (`../models/scripts/`).


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "pandas", "numpy", "matplotlib", "tqdm"])
print("deps ok")


In [ ]:
import os, json, time, subprocess, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

QF = Path.cwd().parent
MODELS_DIR = QF / "models"
ENV = {**os.environ, "PATH": f"{Path.home()}/.local/bin:" + os.environ.get("PATH", "")}
PORT = 8080

# 8-company universe (the real companies behind the anonymized ID benchmark)
TICKERS = ["AMD", "APP", "GM", "KR", "MCK", "NFLX", "PLTR", "SNDK"]
NAMES = {"AMD":"AMD","APP":"AppLovin","GM":"General Motors","KR":"Kroger",
         "MCK":"McKesson","NFLX":"Netflix","PLTR":"Palantir","SNDK":"SanDisk"}
MODEL    = "qwen35-35b-a3b"    # best speed x capability from the ID benchmark
TOP_K    = 3                   # hold the top-3 each month
COST_BPS = 5                   # per-turnover transaction cost (bps)
ELIG     = 252                 # min trading days of history to be eligible
START    = "2019-07-01"        # ~7-year backtest; names enter the universe as they list
print("universe:", [NAMES[t] for t in TICKERS])


In [ ]:
import yfinance as yf
px = yf.download(TICKERS, start="2017-06-01", auto_adjust=True, progress=False)["Close"][TICKERS]  # +2y lookback buffer
rets = px.pct_change()
print("prices:", px.shape, px.index.min().date(), "->", px.index.max().date())
px.tail(3)


In [ ]:
def reb_dates(px, start):
    idx = px.loc[start:].index
    return list(pd.Series(idx).groupby([idx.year, idx.month]).min())

def features_at(px, dt):
    """Trailing price features per eligible stock, using only data <= dt (no look-ahead)."""
    h = px.loc[:dt]; rows = {}
    for t in px.columns:
        s = h[t].dropna()
        if len(s) < ELIG: continue          # not eligible yet (e.g. SNDK before ~2026)
        p = s.iloc[-1]; r = s.pct_change().dropna()
        rows[t] = dict(mom_1m=p/s.iloc[-21]-1, mom_3m=p/s.iloc[-63]-1, mom_6m=p/s.iloc[-126]-1,
                       mom_12m=p/s.iloc[-252]-1, vol_3m=r.iloc[-63:].std()*np.sqrt(252),
                       ma200_gap=p/s.iloc[-200:].mean()-1, dd_6m=p/s.iloc[-126:].max()-1)
    return pd.DataFrame(rows).T

DATES = reb_dates(px, START)
FEATS = {dt: features_at(px, dt) for dt in DATES}
print(len(DATES), "monthly rebalances", DATES[0].date(), "->", DATES[-1].date())
FEATS[DATES[-1]].round(3)


In [ ]:
def serve(mid):
    """Launch llama-server (native GPU) for one model; wait until healthy."""
    log = open(f"/tmp/_bt_{mid}.log", "w")
    p = subprocess.Popen(["bash", str(MODELS_DIR/"scripts"/"serve.sh"), mid, "--ctx-size", "4096"],
                         stdout=log, stderr=subprocess.STDOUT, env=ENV, cwd=str(MODELS_DIR))
    for _ in range(120):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3).status == 200: return p, log
        except Exception: pass
        time.sleep(2)
    raise RuntimeError("server not ready")

def llm_score(feat):
    """Score each ANONYMIZED stock 0-100 for next-month relative return (no names -> no hindsight)."""
    labels = [f"S{i+1}" for i in range(len(feat))]
    tbl = feat.copy(); tbl.index = labels
    prompt = (f"You are a disciplined quantitative equity analyst. Below are {len(feat)} anonymized "
        "stocks (labels only, no identities) with trailing price-based features. Higher momentum = recent "
        "strength; higher vol_3m = risk; ma200_gap = distance vs the 200-day average; dd_6m = drawdown "
        "from the 6-month high.\n\n" + tbl.round(3).to_string() + "\n\nScore each label 0-100 for expected "
        "RELATIVE total return over the next ~1 month (higher = more bullish). Output ONLY a JSON object "
        'like {"S1": 70, "S2": 40}. No other text.')
    body = json.dumps({"messages":[{"role":"user","content":prompt}], "temperature":0.2, "max_tokens":300,
                       "chat_template_kwargs":{"enable_thinking":False}}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=body,
                                 headers={"Content-Type":"application/json"})
    txt = json.load(urllib.request.urlopen(req, timeout=120))["choices"][0]["message"]["content"]
    i, j = txt.find("{"), txt.rfind("}")
    sc = json.loads(txt[i:j+1]) if 0 <= i < j else {}
    return pd.Series({feat.index[k]: float(sc.get(labels[k], 0)) for k in range(len(feat))})

def topk(scores, k=TOP_K):
    top = scores.sort_values(ascending=False).head(k)
    return {t: 1.0/len(top) for t in top.index} if len(top) else {}

print("helpers ready")


In [ ]:
from tqdm.notebook import tqdm
# Momentum + Buy&Hold need no LLM
mom_sched = {dt: topk(0.5*FEATS[dt].mom_3m + 0.5*FEATS[dt].mom_6m) for dt in DATES}
# buy & hold: equal-weight eligible; re-equal-weight only when a new name lists (staggered entry)
bh_sched, _prev = {}, None
for _dt in DATES:
    _elig = list(FEATS[_dt].index)
    if set(_elig) != _prev:
        bh_sched[_dt] = {t: 1.0/len(_elig) for t in _elig}; _prev = set(_elig)

# LLM arm: serve the model once, score every rebalance (anonymized features)
llm_sched = {}
proc, log = serve(MODEL)
try:
    for dt in tqdm(DATES, desc="LLM scoring"):
        f = FEATS[dt]
        try:    s = llm_score(f)
        except Exception as e:
            s = 0.5*f.mom_3m + 0.5*f.mom_6m; print("  fallback", dt.date(), e)   # robust to a bad JSON
        llm_sched[dt] = topk(s)
finally:
    proc.terminate()
    try: proc.wait(timeout=15)
    except Exception: proc.kill()
    log.close()
print("schedules built")


In [ ]:
def backtest(schedule, cost_bps=COST_BPS):
    """Daily portfolio value from a {rebalance_date: {ticker: weight}} schedule; weights drift between rebalances."""
    d0 = min(schedule); idx = rets.loc[d0:].index
    hold = pd.Series(0.0, index=px.columns); pv = 1.0; eq = []
    for dt in idx:
        if hold.sum() > 0:
            hold = hold*(1+rets.loc[dt].fillna(0)); pv = hold.sum()      # accrue the day first (no look-ahead)
        if dt in schedule:
            tgt = pd.Series(schedule[dt], index=px.columns).fillna(0.0)
            cur = hold/pv if pv > 0 else pd.Series(0.0, index=px.columns)
            pv *= 1 - cost_bps/1e4 * (tgt-cur).abs().sum()               # turnover cost
            hold = tgt*pv
        eq.append(pv)
    return pd.Series(eq, index=idx)

curves = {"Buy&Hold": backtest(bh_sched, 0), "Momentum": backtest(mom_sched), "LLM": backtest(llm_sched)}
print("backtested", list(curves))


In [ ]:
import matplotlib.pyplot as plt

def stats(eq):
    r = eq.pct_change().dropna(); yrs = (eq.index[-1]-eq.index[0]).days/365.25
    return dict(total=eq.iloc[-1]-1, cagr=eq.iloc[-1]**(1/yrs)-1, vol=r.std()*np.sqrt(252),
                sharpe=(r.mean()*252)/(r.std()*np.sqrt(252)), maxdd=(eq/eq.cummax()-1).min())

tbl = pd.DataFrame({k: stats(v) for k, v in curves.items()}).T
show = tbl.copy()
for c in ["total","cagr","vol","maxdd"]: show[c] = (show[c]*100).round(1).astype(str) + "%"
show["sharpe"] = show["sharpe"].round(2)
display(show)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for k, v in curves.items(): ax[0].plot(v.index, v, label=k, lw=2)
ax[0].set_yscale("log"); ax[0].set_title("Growth of $1 (log scale)"); ax[0].legend(); ax[0].grid(alpha=.3)
for k, v in curves.items(): ax[1].plot(v.index, (v/v.cummax()-1)*100, label=k, lw=1.5)
ax[1].set_title("Drawdown (%)"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## What this shows

7-year run (Jul 2019 → Jul 2026), universe growing 5 → 8 as names list:

| arm | total | CAGR | Sharpe | max DD |
|---|--:|--:|--:|--:|
| Buy & Hold | +1190% | 44% | 1.30 | −37% |
| **Momentum** | **+3888%** | **69%** | **1.62** | −38% |
| LLM | +865% | 38% | 1.27 | **−24%** |

- A dead-simple **momentum rule won big** — this universe (AMD/Netflix/Palantir) trended hard, and top-3 momentum rode the winners.
- The **LLM was the most defensive** (lowest drawdown by far, −24%) but **trailed even buy-and-hold on return** — it kept rotating into lower-vol names and left the trend on the table.
- Net: the LLM's judgment on raw *price features* delivered **risk reduction, not alpha** — and here that wasn't even enough to beat passive. Honest, common outcome: LLMs aren't quant-timers on numbers alone.

**Where an LLM might actually help** (extensions): feed it *fundamentals* or *news/filings text* (its real strength) instead of price stats; use it for position sizing / risk overlays; or widen the universe so momentum concentration matters less. Numbers vary run-to-run (LLM temperature, fresh data) — re-run to regenerate.
